# 05 · Calidad de datos

**Fase CRISP-DM:** 2. Entendimiento de los datos

**Objetivo:** auditar las tablas del contrato **como si fueran datos reales de Puzzle**, sin mirar la verdad oculta, en seis dimensiones: completitud, unicidad, integridad referencial, validez, consistencia y outliers. Al final, diagnóstico de los **datos faltantes** (MCAR, MAR, MNAR).

**Prueba de fuego:** en el notebook 03 se sembró ruido a propósito (pedidos del mesero sin comensal, cuentas duplicadas, mesas gigantes, onboarding incompleto, calificaciones faltantes no aleatorias). **Un buen diagnóstico tiene que encontrarlo.** La verdad oculta solo se usa al final, para verificar.

## Configuración y carga

In [1]:
import numpy as np
import pandas as pd

from puzzle_recsys.config import load_paths, load_simulation
from puzzle_recsys.quality.checks import (
    completeness,
    duplicate_keys,
    missingness_test,
    orphan_keys,
    out_of_range,
    price_consistency,
    robust_outliers,
)
from puzzle_recsys.repository.parquet_repo import ParquetRepository, run_id_for

paths, sim = load_paths(), load_simulation()
repo = ParquetRepository(paths.synthetic / run_id_for(sim.run_name, sim.seed))
names = [
    "restaurants",
    "menu_items",
    "menu_item_price_history",
    "menu_item_availability_events",
    "diners",
    "diner_profiles",
    "dining_sessions",
    "guests",
    "order_items",
    "dish_ratings",
    "visit_ratings",
]
t = {n: repo.read_table(n) for n in names}
pd.DataFrame(
    {
        "filas": {n: len(df) for n, df in t.items()},
        "columnas": {n: df.shape[1] for n, df in t.items()},
    }
)

,filas,columnas
restaurants,1500,14
menu_items,36115,10
menu_item_price_history,50501,4
menu_item_availability_events,438862,4
diners,20224,2
diner_profiles,20000,5
dining_sessions,514510,4
guests,1479362,4
order_items,3409935,9
dish_ratings,422415,4


## 1. Completitud

Proporción de valores presentes por columna. Solo se muestran las columnas **incompletas**: un nulo puede ser legítimo (una llave opcional) o un problema; esa distinción es la que importa.

In [2]:
rows = []
for name, df in t.items():
    for col, share in completeness(df).items():
        if share < 1:
            rows.append({"tabla": name, "columna": col, "% completo": round(share * 100, 2)})
pd.DataFrame(rows)

,tabla,columna,% completo
0,restaurants,chain,2.73
1,menu_items,archived_at,17.35
2,menu_item_price_history,valid_to,28.49
3,diner_profiles,declared_spice,60.36
4,guests,diner_id,50.70
5,order_items,guest_id,95.99


## 2. Unicidad de las llaves primarias

In [3]:
keys = {
    "restaurants": ["restaurant_id"],
    "menu_items": ["menu_item_id"],
    "diners": ["diner_id"],
    "diner_profiles": ["diner_id"],
    "dining_sessions": ["session_id"],
    "guests": ["guest_id"],
    "order_items": ["order_item_id"],
    "dish_ratings": ["order_item_id"],
    "visit_ratings": ["session_id", "guest_id"],
    "menu_item_price_history": ["menu_item_id", "valid_from"],
}
pd.Series(
    {n: duplicate_keys(t[n], k) for n, k in keys.items()}, name="filas duplicadas"
).to_frame().T

,restaurants,menu_items,diners,diner_profiles,dining_sessions,guests,order_items,dish_ratings,visit_ratings,menu_item_price_history
filas duplicadas,0,0,0,0,0,0,0,0,0,0


## 3. Integridad referencial (llaves foráneas que apuntan a algo que no existe)

In [4]:
fks = [
    ("menu_items", "restaurant_id", "restaurants", "restaurant_id"),
    ("dining_sessions", "restaurant_id", "restaurants", "restaurant_id"),
    ("guests", "session_id", "dining_sessions", "session_id"),
    ("guests", "diner_id", "diners", "diner_id"),
    ("order_items", "session_id", "dining_sessions", "session_id"),
    ("order_items", "guest_id", "guests", "guest_id"),
    ("order_items", "menu_item_id", "menu_items", "menu_item_id"),
    ("dish_ratings", "order_item_id", "order_items", "order_item_id"),
    ("visit_ratings", "session_id", "dining_sessions", "session_id"),
    ("diner_profiles", "diner_id", "diners", "diner_id"),
]
pd.DataFrame(
    [
        {
            "hijo.columna": f"{c}.{ck}",
            "padre": f"{p}.{pk}",
            "huérfanos": orphan_keys(t[c], ck, t[p], pk),
        }
        for c, ck, p, pk in fks
    ]
)

,hijo.columna,padre,huérfanos
0,menu_items.restaurant_id,restaurants.restaurant_id,0
1,dining_sessions.restaurant_id,restaurants.restaurant_id,0
2,guests.session_id,dining_sessions.session_id,0
3,guests.diner_id,diners.diner_id,0
4,order_items.session_id,dining_sessions.session_id,0
5,order_items.guest_id,guests.guest_id,0
6,order_items.menu_item_id,menu_items.menu_item_id,0
7,dish_ratings.order_item_id,order_items.order_item_id,0
8,visit_ratings.session_id,dining_sessions.session_id,0
9,diner_profiles.diner_id,diners.diner_id,0


## 4. Validez (rangos de dominio)

In [5]:
r = t["restaurants"]
validity = {
    "precio de platillo ≤ 0": out_of_range(t["menu_items"]["price_cents"], 1, 10**9),
    "picante fuera de 0 a 3": out_of_range(t["menu_items"]["spice_level"], 0, 3),
    "estrellas fuera de 1 a 5 (platillos)": out_of_range(t["dish_ratings"]["stars"], 1, 5),
    "estrellas fuera de 1 a 5 (visitas)": out_of_range(t["visit_ratings"]["stars"], 1, 5),
    "mesa con menos de 1 persona": out_of_range(t["dining_sessions"]["pax"], 1, 10**6),
    "cantidad pedida < 1": out_of_range(t["order_items"]["quantity"], 1, 10**6),
    "restaurante fuera de la CDMX (lat)": out_of_range(r["lat"], 19.0, 19.6),
    "restaurante fuera de la CDMX (lng)": out_of_range(r["lng"], -99.4, -98.9),
}
pd.Series(validity, name="violaciones").to_frame().T

,precio de platillo ≤ 0,picante fuera de 0 a 3,estrellas fuera de 1 a 5 (platillos),estrellas fuera de 1 a 5 (visitas),mesa con menos de 1 persona,cantidad pedida < 1,restaurante fuera de la CDMX (lat),restaurante fuera de la CDMX (lng)
violaciones,0,0,0,0,0,0,0,0


## 5. Consistencia entre tablas

- ¿Cada línea se cobró al **precio vigente** en el momento del pedido?
- ¿El **tamaño de mesa** (`pax`) coincide con los invitados registrados?
- ¿Ningún pedido es **anterior** a la apertura de su mesa?
- ¿Los pedidos sin comensal vienen **todos** del mesero (`source = WAITER`)?
- ¿Nadie está en **dos mesas a la vez** (menos de 3 h entre mesas) ni aparece dos veces en la misma?

In [6]:
s, g, oi = t["dining_sessions"], t["guests"], t["order_items"]
opened = s.set_index("session_id")["opened_at"]
guests_per_table = g.groupby("session_id").size().reindex(s["session_id"]).to_numpy()
no_guest = oi["guest_id"].isna()
seated = g.dropna(subset=["diner_id"]).merge(s[["session_id", "opened_at"]], on="session_id")
seated = seated.sort_values(["diner_id", "opened_at"])
gap_h = seated.groupby("diner_id")["opened_at"].diff().dt.total_seconds() / 3600
pd.Series(
    {
        "% líneas al precio vigente": price_consistency(oi, t["menu_item_price_history"]) * 100,
        "mesas con pax ≠ invitados": int((guests_per_table != s["pax"].to_numpy()).sum()),
        "pedidos antes de abrir la mesa": int(
            (oi["created_at"] < oi["session_id"].map(opened)).sum()
        ),
        "líneas sin comensal": int(no_guest.sum()),
        "…de esas, cargadas por el mesero (%)": float(
            (oi.loc[no_guest, "source"] == "WAITER").mean() * 100
        ),
        "personas en dos mesas a < 3 h": int(
            (gap_h < sim.visits["min_hours_between_visits"]).sum()
        ),
        "personas repetidas en una mesa": int(
            seated.duplicated(["session_id", "diner_id"]).sum()
        ),
    }
).round(2).to_frame("valor").T

,% líneas al precio vigente,mesas con pax ≠ invitados,pedidos antes de abrir la mesa,líneas sin comensal,"…de esas, cargadas por el mesero (%)",personas en dos mesas a < 3 h,personas repetidas en una mesa
valor,100.0,0.0,0.0,136884.0,100.0,0.0,0.0


## 6. Outliers

Con el **z-score modificado** (mediana y MAD, umbral 3.5): no supone normalidad y no se deja arrastrar por los propios outliers. Se revisan el tamaño de mesa y el precio por tipo de plato.

In [7]:
pax_flags = robust_outliers(s["pax"])
display(
    pd.Series(
        {
            "mesas atípicas": int(pax_flags.sum()),
            "% de mesas": round(pax_flags.mean() * 100, 3),
            "pax mínimo entre las atípicas": int(s.loc[pax_flags, "pax"].min()),
            "pax máximo": int(s["pax"].max()),
        }
    )
    .to_frame("valor")
    .T
)
mi = t["menu_items"]
mi.assign(atipico=mi.groupby("course")["price_cents"].transform(robust_outliers)).groupby("course")[
    "atipico"
].mean().mul(100).round(2).to_frame("% precios atípicos").T

,mesas atípicas,% de mesas,pax mínimo entre las atípicas,pax máximo
valor,1040.0,0.202,12.0,30.0


course,DESSERT,DRINK,MAIN,SIDE,STARTER
% precios atípicos,0.42,3.72,2.8,4.27,0.97


## 7. Diagnóstico de datos faltantes

Tres mecanismos (Rubin):

- **MCAR:** falta completamente al azar. Ignorarlo no sesga.
- **MAR:** falta según **otra variable observada**. Se corrige condicionando en esa variable.
- **MNAR:** falta según **el propio valor que falta**. Es el más peligroso y **no se puede comprobar con los datos observados**.

Para cada faltante se compara la distribución de una variable observada entre filas con y sin el dato (Kolmogorov-Smirnov o chi-cuadrada).

In [8]:
lines_rated = oi.assign(
    stars=oi["order_item_id"].map(t["dish_ratings"].set_index("order_item_id")["stars"])
)
registered = oi["guest_id"].isin(g.loc[g["diner_id"].notna(), "guest_id"])
profiles = t["diner_profiles"].assign(declared=lambda d: np.where(d["completed"], 1.0, np.nan))
visits_per_diner = g.dropna(subset=["diner_id"]).groupby("diner_id").size()
profiles["visitas"] = profiles["diner_id"].map(visits_per_diner).fillna(0)
oi_hour = oi.assign(
    hora=oi["created_at"].dt.tz_convert(sim.calendar["timezone"]).dt.hour, guest=oi["guest_id"]
)
pd.DataFrame(
    [
        {
            "faltante": "comensal de la línea (guest_id)",
            "comparado contra": "precio",
            "p-valor": missingness_test(oi_hour, "guest", "unit_price_cents"),
        },
        {
            "faltante": "comensal de la línea (guest_id)",
            "comparado contra": "hora",
            "p-valor": missingness_test(oi_hour, "guest", "hora"),
        },
        {
            "faltante": "onboarding",
            "comparado contra": "visitas al año",
            "p-valor": missingness_test(profiles, "declared", "visitas"),
        },
        {
            "faltante": "calificación del platillo",
            "comparado contra": "precio",
            "p-valor": missingness_test(lines_rated[registered], "stars", "unit_price_cents"),
        },
    ]
).assign(
    lectura=lambda d: np.where(
        d["p-valor"] < 0.01, "depende de esa variable (no es MCAR)", "consistente con MCAR"
    )
).round(4)

,faltante,comparado contra,p-valor,lectura
0,comensal de la línea (guest_id),precio,0.5690,consistente con MCAR
1,comensal de la línea (guest_id),hora,1.0000,consistente con MCAR
2,onboarding,visitas al año,0.5997,consistente con MCAR
3,calificación del platillo,precio,0.0000,depende de esa variable (no es MCAR)


### Verificación con la verdad oculta (solo posible porque es una simulación)

La calificación se diseñó **MNAR**: califica más quien quedó **muy** contento o **muy** molesto. Con los datos observados eso **no** se puede comprobar, pero aquí sí tenemos la utilidad vivida de cada línea.

In [9]:
truth = repo.ground_truth.read_table("order_item_utilities").set_index("order_item_id")[
    "experienced"
]
z = (truth - truth.mean()) / truth.std()
rated = lines_rated.loc[registered, ["order_item_id", "stars"]].assign(
    extremo=lambda d: d["order_item_id"].map(z.abs())
)
by = (
    rated.assign(califico=rated["stars"].notna())
    .groupby(
        pd.qcut(rated["extremo"], 4, labels=["Q1 (tibio)", "Q2", "Q3", "Q4 (extremo)"]),
        observed=True,
    )["califico"]
    .mean()
)
by.mul(100).round(1).to_frame("% que calificó").T

extremo,Q1 (tibio),Q2,Q3,Q4 (extremo)
% que calificó,14.9,19.2,25.2,38.7


## 8. Cuentas duplicadas

Una persona que se registró dos veces (con correo y con celular) aparece como **dos comensales** con el historial partido. Con reglas de calidad **no se puede detectar** sin datos de identidad (correo, teléfono). En Puzzle se previene en el origen con la restricción de unicidad de `DinerIdentity(type, value)`. Aquí solo se cuantifica con la verdad oculta.

In [10]:
dups = repo.ground_truth.read_table("duplicate_accounts")
pd.Series(
    {
        "cuentas duplicadas": len(dups),
        "% de comensales afectados": round(len(dups) / len(t["diners"]) * 100, 2),
    }
).to_frame("valor").T

,cuentas duplicadas,% de comensales afectados
valor,224.0,1.11


## Conclusiones

**Volumen auditado:** 11 tablas; 514,510 mesas, 1,479,362 invitados, 3,409,935 líneas de pedido y 20,224 cuentas de comensal (20,000 personas más las cuentas duplicadas).

**Reglas estructurales: todo limpio.** 0 llaves duplicadas, 0 llaves foráneas huérfanas y 0 violaciones de dominio. El 100% de las líneas se cobró al precio vigente en el momento del pedido; ninguna mesa tiene `pax` distinto de sus invitados y ningún pedido es anterior a la apertura de su mesa, y nadie está en dos mesas a menos de 3 h (este chequeo se agregó después de que el EDA del notebook 06 encontrara el problema; ver notebook 03, corrección 6).

**El diagnóstico encontró todo el ruido sembrado en el notebook 03:**

| Ruido sembrado | Cómo se detectó | Resultado |
|---|---|---|
| Pedidos del mesero sin comensal | Completitud + consistencia | 136,884 líneas sin `guest_id` (4.0%), **100%** con `source = WAITER` |
| Mesas gigantes | z-score modificado | 1,040 mesas atípicas (0.20%), de 12 a 30 personas |
| Onboarding incompleto | Completitud + prueba KS | `declared_spice` 60% completo; faltante independiente de las visitas (p = 0.60): **MCAR** |
| Calificaciones no aleatorias | Prueba KS contra el precio | p < 0.001: **no es MCAR** |
| Cuentas duplicadas | Solo con la verdad oculta | 224 cuentas (1.11% de los comensales) |

**Nulos legítimos, no problemas:** `guests.diner_id` (51%: invitados anónimos), `menu_items.archived_at` y `menu_item_price_history.valid_to` (platillos vigentes), `restaurants.chain` (independientes).

**Calificaciones (MNAR).** Con los datos observados solo se puede decir que **no son MCAR**. La verdad oculta confirma el mecanismo: la tasa de calificación sube con lo extremo de la experiencia (14.9% → 19.2% → 25.2% → 38.7%). **Implicación para el modelo:** las estrellas están sesgadas hacia los extremos, así que **no se usan como etiqueta principal**; la señal de relevancia será el pedido (implícito) y las estrellas solo como señal auxiliar.

**Implicaciones para la preparación de datos (notebook 09):**
- Las líneas sin comensal **no se descartan**: cuentan para la popularidad del platillo y para la lista "para la mesa", pero no para el historial individual ("para ti").
- Las mesas gigantes se conservan como datos reales, pero se acota su peso (son eventos, no hábitos).
- El onboarding faltante es MCAR: imputarlo con una categoría "no declarado" no introduce sesgo.
- Las cuentas duplicadas no se pueden resolver sin datos de identidad; en Puzzle se previenen en el origen con la unicidad de `DinerIdentity(type, value)`. Aquí se aceptan como ruido (1%).

**Precios atípicos por tipo de plato** (0.4% a 4.3%) son platillos premium o menús de alta gama, no errores: el precio pasó todas las reglas de validez y consistencia.
